In [1]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BYlUHaillwM8EUItaIytHQ/companypolicies.txt"

--2026-06-03 15:31:52--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BYlUHaillwM8EUItaIytHQ/companypolicies.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15660 (15K) [text/plain]
Saving to: ‘companypolicies.txt.3’

companypolicies.txt 100%[===================>]  15.29K  --.-KB/s    in 0.05s   

2026-06-03 15:31:53 (285 KB/s) - ‘companypolicies.txt.3’ saved [15660/15660]



In [2]:
from langchain_community.document_loaders import TextLoader

In [3]:
loader = TextLoader("companypolicies.txt")
data = loader.load() 
data[0].page_content

"1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violati

In [4]:
#Split the data 
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100, 
    chunk_overlap = 20,
    length_function = len
)
texts = text_splitter.split_documents(data)
print(len(texts))

215


In [5]:
#Embedding model 
from langchain_aws import BedrockEmbeddings 
embedding_model = BedrockEmbeddings(
    model_id = "amazon.titan-embed-text-v2:0", 
    region_name = "us-east-1"
)
response = embedding_model.embed_query("This is a demo query")
print(len(response))

1024


In [6]:
#Vector store
from langchain_community.vectorstores import Chroma
vector_db = Chroma.from_documents(texts,embedding_model)
print(vector_db._collection.count())

215


In [8]:
#Similarity search 
query = "Harrasment policy"
docs = vector_db.similarity_search(query,k = 2)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='8.\tAnti-discrimination and Harassment Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Harassment: Harassment in any form, whether based on the aforementioned characteristics or any')]

In [10]:
#Faiss vector db 
from langchain_community.vectorstores import FAISS
faissdb = FAISS.from_documents(texts, embedding_model)
faiss_docs = faissdb.similarity_search(query, k =2)
faiss_docs

[Document(id='5fa83cc9-5abc-446b-a878-b4b2be3d998f', metadata={'source': 'companypolicies.txt'}, page_content='8.\tAnti-discrimination and Harassment Policy'),
 Document(id='49af4f5c-5f48-43a2-a852-6af680563585', metadata={'source': 'companypolicies.txt'}, page_content='Harassment: Harassment in any form, whether based on the aforementioned characteristics or any')]

In [11]:
text = "Instructlab is the best open source tool for fine-tuning a LLM."

In [12]:
from langchain_core.documents import Document

In [13]:
new_chunk =  Document(
    page_content=text,
    metadata={
        "source": "ibm.com",
        "page": 1
    }
)

In [14]:
new_chunks = [new_chunk]

In [16]:
print(vector_db._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [17]:
vector_db.add_documents(
    new_chunks,
    ids=["215"]
)

['215']

In [19]:
vector_db._collection.count()

216

In [20]:
print(vector_db._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['Instructlab is the best open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'ibm.com'}]}


In [21]:
update_chunk =  Document(
    page_content="Instructlab is a perfect open source tool for fine-tuning a LLM.",
    metadata={
        "source": "ibm.com",
        "page": 1
    }
)

In [22]:
vector_db.update_document(
    '215',
    update_chunk,
)

In [23]:
print(vector_db._collection.get(ids=['215']))

{'ids': ['215'], 'embeddings': None, 'documents': ['Instructlab is a perfect open source tool for fine-tuning a LLM.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 1, 'source': 'ibm.com'}]}


In [25]:
vector_db._collection.delete(ids=['215'])

In [27]:
print(vector_db._collection.get(ids=['215']))

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}
